# CryoDRGN: Reconstructing Heterogeneous Structures from Cryo-EM Images with Neural Networks

*A tutorial notebook for `xulabs/edu`*

**Paper:** Zhong, E.D., Bepler, T., Berger, B. & Davis, J.H. *CryoDRGN: reconstruction of
heterogeneous cryo-EM structures using neural networks.* **Nature Methods** 18, 176–185 (2021).
A preliminary version appeared at ICLR 2020. Code: [github.com/ml-struct-bio/cryodrgn](https://github.com/ml-struct-bio/cryodrgn)

## Motivation

Cryo-electron microscopy (cryo-EM) single-particle analysis reconstructs a 3D structure from
tens of thousands of noisy 2D projection images of individual molecules, frozen in random
orientations. Classical reconstruction algorithms (as in RELION, cryoSPARC) assume every
particle is a noisy view of the **same rigid structure** and solve for one 3D density map.

The problem: most interesting biomolecules are not rigid. They exist as mixtures of discrete
states (**compositional heterogeneity** — a subunit present in some particles, absent in
others) and/or continuous conformational ensembles (**conformational heterogeneity** — a
domain swinging, a loop flexing). Averaging all particles together under the rigid-structure
assumption blurs out exactly this information — the biologically interesting part.

CryoDRGN's idea: replace the single 3D density map with a **neural network that outputs
density as a function of both 3D position *and* a learned per-particle latent code** — so
each particle can be explained by its own point in a continuous "conformation space",
learned end-to-end from the same 2D images a classical pipeline would otherwise collapse
into one blurry average.

## What this notebook covers

1. The cryo-EM forward model: how a 3D structure becomes a 2D image (the **projection-slice
   theorem**) and how the microscope's optics distort it (the **CTF**).
2. Why cryoDRGN represents images and volumes via the **Hartley transform** rather than the
   (complex) Fourier transform.
3. The architecture: an **encoder** mapping images to a latent code, and a **coordinate-based
   decoder** mapping (frequency-space position, latent code) → density — implemented here
   from scratch in NumPy, with every gradient written out by hand.
4. A synthetic two-blob "molecule" with one blob sliding continuously (an idealized hinge
   motion) as a fully self-contained, CPU-friendly stand-in for real conformational
   heterogeneity, on which we train the model and check whether it recovers the true motion
   from images alone.
5. What happens if you ignore the heterogeneity and just average everything (spoiler: you
   lose the interesting part).
6. Notes on running real cryoDRGN on real EMPIAR data, and discussion questions.

**Runtime:** the full training loop runs in a few minutes on a laptop CPU. No GPU, and no
external ML framework, is required — every neural network layer and its backward pass is
implemented here in plain NumPy, so this can run in Colab exactly as-is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import rotate as nd_rotate

# House dark theme
plt.rcParams.update({
    "figure.facecolor": "#1e1e1e", "axes.facecolor": "#1e1e1e", "savefig.facecolor": "#1e1e1e",
    "axes.edgecolor": "white", "axes.labelcolor": "white", "xtick.color": "white",
    "ytick.color": "white", "text.color": "white", "axes.titlecolor": "white",
    "grid.color": "#444444", "figure.dpi": 100,
})

rng_global = np.random.default_rng(0)

## 1. The cryo-EM forward model

### 1.1 The projection-slice theorem

A cryo-EM image is (to good approximation) a **projection** of a 3D density $V(\mathbf{r})$
along the electron beam direction. Let $R \in SO(3)$ be the (a priori unknown, but for our
purposes assumed already solved for) orientation of the particle. The projection-slice
theorem states:

$$
\mathcal{F}_{2D}\big[\, \text{Proj}_{R}(V) \,\big](\mathbf{k}_{2D})
\;=\; \mathcal{F}_{3D}[V]\big(R \cdot [\mathbf{k}_{2D}, 0]^\top\big)
$$

In words: the 2D Fourier transform of the projection image equals a **planar slice through
the origin** of the 3D Fourier transform of the volume, oriented perpendicular to the viewing
direction and picked out by $R$. This is the single most important fact in cryo-EM
reconstruction — it is *why* combining many 2D images at many orientations can determine a
full 3D structure, and it is exactly the mechanism cryoDRGN's coordinate-based decoder
exploits: instead of storing a 3D grid, the decoder is queried *only* at the 3D coordinates
that lie on a given particle's central slice.

### 1.2 Why the Hartley transform, not the Fourier transform

CryoDRGN works with the real-valued **discrete Hartley transform (DHT)** instead of the
complex Fourier transform:

$$
H(\mathbf{k}) = \mathrm{Re}\{F(\mathbf{k})\} - \mathrm{Im}\{F(\mathbf{k})\}
$$

The projection-slice theorem holds equally for the Hartley transform. Because $H$ is
real-valued, both the encoder's input and the decoder's output can be plain real numbers —
no complex arithmetic needs to flow through the network. Given a real image's Hartley
transform, the original Fourier coefficients are still fully recoverable via the even/odd
decomposition $\mathrm{Re}\{F(\mathbf{k})\} = \tfrac{1}{2}[H(\mathbf{k}) + H(-\mathbf{k})]$,
$\mathrm{Im}\{F(\mathbf{k})\} = \tfrac{1}{2}[H(-\mathbf{k}) - H(\mathbf{k})]$, which we use
below to go back and forth between Hartley coefficients and real-space images.

In [ ]:
def gaussian_blob_hartley(k, mu, sigma, amp):
    """Closed-form 3D Hartley transform of a single isotropic Gaussian blob.

    Real-space blob:  amp * exp(-||r - mu||^2 / (2 sigma^2))
    Fourier pair:      F(k) = amp (2*pi*sigma^2)^1.5 * exp(-2 pi^2 sigma^2 |k|^2)
                                * exp(-2*pi*i*k.mu)
    Hartley:            H(k) = Re(F(k)) - Im(F(k))

    Using an analytic (Gaussian-blob) toy molecule lets us evaluate the *exact*
    Hartley transform of the "structure" at any 3D frequency coordinate directly,
    with no 3D grid, no 3D FFT, and no interpolation -- mirroring how cryoDRGN's
    decoder is itself a continuous, coordinate-based function of frequency-space
    position.

    Args:
        k: (..., 3) array of frequency-space coordinates, 1/Angstrom
        mu: (3,) blob center, Angstrom
        sigma: blob width, Angstrom
        amp: blob amplitude (arbitrary density units)

    Returns:
        (...,) array of real-valued Hartley coefficients
    """
    k2 = np.sum(k ** 2, axis=-1)
    envelope = amp * (2 * np.pi * sigma ** 2) ** 1.5 * np.exp(-2 * np.pi ** 2 * sigma ** 2 * k2)
    phase = 2 * np.pi * np.sum(k * mu, axis=-1)
    return envelope * (np.cos(phase) + np.sin(phase))


def volume_hartley(k, blobs):
    """Hartley transform of a multi-blob volume (Hartley transform is linear).

    Args:
        k: (..., 3) frequency coordinates
        blobs: list of dicts with keys 'mu', 'sigma', 'amp'
    """
    out = np.zeros(k.shape[:-1])
    for b in blobs:
        out = out + gaussian_blob_hartley(k, np.asarray(b["mu"]), b["sigma"], b["amp"])
    return out


def make_conformation(t, blob_a=None, blob_b_start=None, blob_b_end=None, sigma=9.0):
    """Our toy 'molecule': two Gaussian blobs. Blob A is fixed; blob B slides
    linearly from `blob_b_start` to `blob_b_end` as the conformational
    coordinate t goes from 0 to 1 -- a simple stand-in for a real hinge
    motion / domain swing, which is the kind of continuous heterogeneity
    cryoDRGN is built to recover.
    """
    blob_a = blob_a if blob_a is not None else np.array([-18.0, 0.0, 0.0])
    blob_b_start = blob_b_start if blob_b_start is not None else np.array([14.0, -16.0, 0.0])
    blob_b_end = blob_b_end if blob_b_end is not None else np.array([14.0, 16.0, 0.0])
    mu_b = (1 - t) * blob_b_start + t * blob_b_end
    return [
        {"mu": blob_a, "sigma": sigma, "amp": 1.0},
        {"mu": mu_b, "sigma": sigma, "amp": 1.0},
    ]

In [ ]:
# Sanity-check / visualize the ground-truth trajectory: direct analytic real-space
# projections (sum of blobs integrated along the viewing axis), no FFT machinery involved.
def direct_projection(t, n=24, apix=3.0):
    lin = (np.arange(n) - n // 2) * apix
    xs, ys = np.meshgrid(lin, lin, indexing="xy")
    blobs = make_conformation(t)
    img = np.zeros((n, n))
    for b in blobs:
        mu = b["mu"]
        img += b["amp"] * 2 * np.pi * b["sigma"] ** 2 * np.exp(
            -((xs - mu[0]) ** 2 + (ys - mu[1]) ** 2) / (2 * b["sigma"] ** 2)
        )
    return img

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t in zip(axes, [0.0, 0.25, 0.5, 0.75, 1.0]):
    ax.imshow(direct_projection(t), cmap="gray")
    ax.set_title(f"t = {t}")
    ax.axis("off")
plt.suptitle("Ground-truth conformational trajectory: blob B slides past blob A", y=1.05)
plt.tight_layout()
plt.show()

## 2. Central-slice sampling and the Hartley $\leftrightarrow$ real-space round trip

To generate a particle image at pose $R$ we:

1. Build the 2D image-plane frequency lattice $(u, v)$ and embed it in 3D as $(u, v, 0)$.
2. Rotate by $R$ to get the 3D frequency coordinates $\mathbf{k} = R \cdot (u, v, 0)^\top$
   that the projection-slice theorem says this pose's image transform lives on.
3. Evaluate the volume's (analytic, in our toy case) Hartley transform at those coordinates.
4. Invert the Hartley transform to get a real-space image.

All of `slice_coords`, `hartley_to_real`, and `real_to_hartley` below are written for a
**centered** array convention (index `N//2` $\leftrightarrow$ zero-frequency / box center),
matching how we build all our coordinate grids — this bookkeeping is the single easiest place
to introduce a bug in any cryo-EM code (FFT libraries default to a very different, *natural*
index-0-at-the-origin ordering), so we verify the round trip numerically below before trusting
any of it.

In [ ]:
def slice_coords(n, apix, R):
    """3D frequency-space coordinates of the central slice picked out by pose R
    (the projection-slice theorem, made concrete).

    Returns:
        k3: (n, n, 3) array of 3D frequency coordinates, 1/Angstrom
    """
    freq = np.fft.fftshift(np.fft.fftfreq(n, d=apix))  # 1/Angstrom, centered
    u, v = np.meshgrid(freq, freq, indexing="xy")
    plane = np.stack([u, v, np.zeros_like(u)], axis=-1)  # (n, n, 3)
    return plane @ R.T


def real_to_hartley(img):
    """Forward discrete Hartley transform of a real-space image via FFT.
    `img` is a centered spatial array; returns a centered frequency array."""
    f = np.fft.fft2(np.fft.ifftshift(img, axes=(-2, -1)))
    h = np.real(f) - np.imag(f)
    return np.fft.fftshift(h, axes=(-2, -1))


def hartley_to_real(h):
    """Inverse Hartley transform via the even/odd decomposition
    Re(F(k)) = [H(k)+H(-k)]/2, Im(F(k)) = [H(-k)-H(k)]/2, then inverse FFT.
    Both input and output use the centered array convention."""
    h0 = np.fft.ifftshift(h, axes=(-2, -1))
    h0_neg = np.roll(np.flip(h0, axis=(-2, -1)), shift=(1, 1), axis=(-2, -1))  # H(-k)
    re = 0.5 * (h0 + h0_neg)
    im = 0.5 * (h0_neg - h0)
    img = np.fft.ifft2(re + 1j * im)
    return np.real(np.fft.fftshift(img, axes=(-2, -1)))


def random_rotation(rng):
    """Uniform random 3D rotation via QR decomposition of a Gaussian random
    matrix (a standard way to sample Haar-random SO(3))."""
    a = rng.normal(size=(3, 3))
    q, r = np.linalg.qr(a)
    q = q * np.sign(np.diag(r))
    if np.linalg.det(q) < 0:
        q[:, 0] *= -1
    return q


def random_inplane_rotation(rng):
    """Random rotation about the viewing (z) axis only -- i.e. a fixed viewing
    direction with a random in-plane particle orientation. A much easier,
    1-degree-of-freedom pose-ambiguity regime than full random SO(3), used
    below to keep the from-scratch NumPy demo trainable in a few minutes on
    a laptop CPU. Real cryoDRGN trains on full random 3D orientations across
    tens-to-hundreds of thousands of particles on a GPU; we return to this
    point in the discussion questions.
    """
    theta = rng.uniform(0, 2 * np.pi)
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

In [ ]:
# --- Verify the round trip and the central-slice formula numerically ---
x0 = rng_global.normal(size=(24, 24))
x1 = hartley_to_real(real_to_hartley(x0))
print("Hartley forward/inverse round-trip max abs error:", np.max(np.abs(x0 - x1)))

# central-slice (analytic) vs. direct real-space projection, identity pose
n, apix = 24, 3.0
blobs = make_conformation(0.7)
h_ideal = volume_hartley(slice_coords(n, apix, np.eye(3)), blobs)
recon = hartley_to_real(h_ideal)
direct = direct_projection(0.7, n=n, apix=apix)
print("correlation(central-slice reconstruction, direct projection):",
      np.corrcoef(recon.ravel(), direct.ravel())[0, 1])

## 3. The contrast transfer function (CTF)

The electron microscope's optics do not pass all spatial frequencies faithfully. Out-of-focus
imaging (deliberately used for phase contrast) modulates each frequency by the CTF:

$$
\gamma(k) = \frac{\pi}{2} C_s \lambda^3 k^4 - \pi \lambda \Delta f\, k^2, \qquad
\mathrm{CTF}(k) = \sqrt{1-a^2}\,\sin\gamma(k) + a\cos\gamma(k)
$$

where $\Delta f$ is the defocus, $C_s$ the spherical aberration coefficient, $\lambda$ the
electron wavelength (a function of accelerating voltage), and $a$ the amplitude-contrast
fraction. This is the same functional form used by RELION, cryoSPARC, and cryoDRGN's own
`ctf.py`. Because the CTF passes through zero and flips sign repeatedly with increasing
frequency, no single image contains the complete signal — this is part of why combining many
images (at different, ideally uncorrelated defocus values) is essential, and it's applied
directly to the (real-valued) Hartley coefficients here, exactly as in the real pipeline.

In [ ]:
def ctf(k2d_mag, defocus, voltage_kv=300.0, cs_mm=2.7, amp_contrast=0.1, bfactor=60.0):
    """Simplified (isotropic, no astigmatism) contrast transfer function.

    Args:
        k2d_mag: spatial frequency magnitude, 1/Angstrom
        defocus: defocus, Angstrom (positive = underfocus)
        voltage_kv: accelerating voltage, kV
        cs_mm: spherical aberration, mm
        amp_contrast: amplitude contrast fraction
        bfactor: envelope B-factor, Angstrom^2 (signal decay at high resolution)
    """
    voltage_v = voltage_kv * 1000.0
    lam = 12.2639 / np.sqrt(voltage_v + 0.97845e-6 * voltage_v ** 2)  # electron wavelength, Angstrom
    cs_ang = cs_mm * 1e7
    gamma = (np.pi / 2.0) * cs_ang * lam ** 3 * k2d_mag ** 4 - np.pi * lam * defocus * k2d_mag ** 2
    envelope = np.exp(-bfactor * k2d_mag ** 2 / 4.0)
    return envelope * (np.sqrt(1 - amp_contrast ** 2) * np.sin(gamma) + amp_contrast * np.cos(gamma))


# Plot the CTF curve for a couple of defocus values
freqs = np.linspace(0, 0.25, 400)  # 1/Angstrom
fig, ax = plt.subplots(figsize=(6, 3.5))
for df, color in [(8000, "#4fc3f7"), (16000, "#ff8a65")]:
    ax.plot(freqs, ctf(freqs, defocus=df), label=f"defocus = {df/1e4:.1f} um", color=color)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("spatial frequency (1/Angstrom)"); ax.set_ylabel("CTF(k)")
ax.legend(); ax.set_title("Contrast transfer function")
plt.tight_layout(); plt.show()

## 4. The full forward model: from 3D structure to noisy particle image

Putting it together, a simulated particle image is generated as:

$$
\text{image} = \mathcal{H}^{-1}\Big[\ \mathrm{CTF}(\mathbf{k})\cdot H_{\text{ideal}}(\mathbf{k})\ \Big] + \text{noise}
$$

where $H_{\text{ideal}}$ is the volume's Hartley transform restricted to this particle's
central slice, and $\mathcal{H}^{-1}$ is the inverse Hartley transform. This is exactly the
generative process cryoDRGN's decoder is trained to invert (up to the CTF and noise, which
are known/estimated per-particle, not something the network has to infer).

In [ ]:
rng = np.random.default_rng(1)
n, apix = 24, 3.0
R = np.eye(3)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, t in enumerate([0, 0.25, 0.5, 0.75, 1.0]):
    blobs = make_conformation(t)
    k3 = slice_coords(n, apix, R)
    h_ideal = volume_hartley(k3, blobs)
    kmag = np.linalg.norm(k3[..., :2], axis=-1)
    c = ctf(kmag, defocus=12000)
    h_obs = h_ideal * c
    clean = hartley_to_real(h_ideal); clean /= np.abs(clean).max() + 1e-8
    img = hartley_to_real(h_obs); img /= np.abs(img).max() + 1e-8
    noisy = img + rng.normal(scale=0.15, size=img.shape)
    axes[0, i].imshow(clean, cmap="gray"); axes[0, i].set_title(f"clean, t={t}"); axes[0, i].axis("off")
    axes[1, i].imshow(noisy, cmap="gray"); axes[1, i].set_title(f"noisy + CTF, t={t}"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

Notice the noisy+CTF row shows the blobs as **dark** against a lighter background — this is
the characteristic contrast-inversion of underfocus phase-contrast imaging, not a bug; it is
exactly what real cryo-EM micrographs look like before CTF correction.

## 5. The cryoDRGN model

CryoDRGN is a **variational autoencoder (VAE)** with an unusual decoder:

* **Encoder** $q_\phi(\mathbf{z} \mid X)$: maps a particle image $X$ (in Hartley-transformed
  form) to a Gaussian posterior over a low-dimensional latent code $\mathbf{z}$
  (its heterogeneity coordinate).
* **Decoder** $p_\theta(H \mid \mathbf{k}, \mathbf{z})$: a **coordinate-based MLP** — given a
  3D frequency-space coordinate $\mathbf{k}$ *and* the latent code $\mathbf{z}$, it outputs
  the predicted Hartley coefficient at that point. Because it's a continuous function of
  $\mathbf{k}$ rather than a fixed voxel grid, it can be queried at exactly the (rotated)
  central-slice coordinates any given particle's known pose picks out — and, after training,
  queried on a full 3D grid at any $\mathbf{z}$ to produce an explicit output volume.

Coordinates are first lifted through a **positional encoding**
(as in NeRF; cryoDRGN uses the same trick) — plain MLPs are spectrally biased toward
low-frequency functions, so raw $(k_x, k_y, k_z)$ is expanded into
$[\mathbf{k}, \sin(2^l \pi \mathbf{k}), \cos(2^l \pi \mathbf{k})]_{l=0}^{L-1}$
before being fed to the decoder, giving it the capacity to represent sharper structural detail.

Training maximizes the evidence lower bound (ELBO):

$$
\mathcal{L} = \underbrace{\big\| \mathrm{CTF}(\mathbf{k})\cdot D_\theta(\mathbf{k}, \mathbf{z}) - H_{\text{obs}}(\mathbf{k}) \big\|^2}_{\text{reconstruction}}
\;+\; \beta \underbrace{D_{KL}\big[q_\phi(\mathbf{z}\mid X) \,\|\, \mathcal{N}(0, I)\big]}_{\text{regularization}}
$$

with $\mathbf{z}$ drawn via the reparameterization trick
$\mathbf{z} = \mu_\phi(X) + \epsilon \cdot \sigma_\phi(X)$, $\epsilon \sim \mathcal{N}(0, I)$,
and the decoder queried at every pixel of the particle's own known central slice, CTF-weighted
before comparison to the observed image's Hartley transform — poses and CTF parameters are
assumed already estimated (from a prior consensus homogeneous refinement), exactly as in the
real cryoDRGN pipeline.

### A scope note on poses, before we train anything

Real cryoDRGN is trained on particles at **fully random 3D orientations** ($SO(3)$), using
tens of thousands to millions of particles and GPU training for hours. When we tried that
full regime here — from-scratch NumPy, CPU, a couple of minutes of compute, a few hundred
particles — the latent code failed to cleanly recover the true conformational coordinate: the
encoder cannot reliably disentangle *pose* from *conformation* from so little data with such
a simple architecture. This is a genuine, instructive finding, not a shortcut we're hiding:
full-pose disentanglement is exactly the hard part of the real method, and it is why real
cryoDRGN needs the dataset sizes and compute it does.

To keep this notebook trainable on a laptop CPU in a few minutes while still faithfully
exercising the *same* central-slice, CTF-weighted, coordinate-decoder machinery, we restrict
poses to random **in-plane rotation only** (one rotational degree of freedom, a fixed viewing
direction). This is still a real pose-disentanglement problem — the network must learn to
be invariant to the unknown in-plane angle while extracting the conformational signal — just
an easier one. Every function above (`slice_coords`, `ctf`, ...) is written for the fully
general case; swapping `random_inplane_rotation` for `random_rotation` in the dataset
generator below is the only change needed to run the harder regime yourself, given more data
and compute (see the discussion questions).

In [ ]:
class Dense:
    """Fully-connected layer: y = x @ W + b, with hand-written backward pass and Adam."""

    def __init__(self, in_dim, out_dim, rng):
        self.W = rng.normal(0, np.sqrt(2.0 / in_dim), size=(in_dim, out_dim))
        self.b = np.zeros(out_dim)
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b); self.vb = np.zeros_like(self.b)
        self.t = 0

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

    def step(self, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for p, dp, m_name, v_name in [(self.W, self.dW, "mW", "vW"), (self.b, self.db, "mb", "vb")]:
            m, v = getattr(self, m_name), getattr(self, v_name)
            m[:] = beta1 * m + (1 - beta1) * dp
            v[:] = beta2 * v + (1 - beta2) * (dp ** 2)
            m_hat, v_hat = m / (1 - beta1 ** self.t), v / (1 - beta2 ** self.t)
            p -= lr * m_hat / (np.sqrt(v_hat) + eps)


class Tanh:
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out

    def backward(self, dout):
        return dout * (1 - self.out ** 2)

    def step(self, *a, **kw):
        pass


class MLP:
    """Sequential stack of layers."""

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def step(self, lr):
        for layer in self.layers:
            layer.step(lr)


def positional_encoding(k, n_freqs=6):
    """NeRF/cryoDRGN-style Fourier feature encoding of coordinates.

    Returns (..., 3 + 3*2*n_freqs): [k, sin(2^0 pi k), cos(2^0 pi k), ...].
    """
    bands = [k]
    for l in range(n_freqs):
        freq = (2.0 ** l) * np.pi
        bands.append(np.sin(freq * k))
        bands.append(np.cos(freq * k))
    return np.concatenate(bands, axis=-1)

In [ ]:
# --- Verify the hand-written backward pass against numerical gradients ---
net = MLP([Dense(4, 8, rng_global), Tanh(), Dense(8, 3, rng_global)])
x = rng_global.normal(size=(5, 4))
target = rng_global.normal(size=(5, 3))

y = net.forward(x)
net.backward((y - target) / y.size)

layer, i, j, eps = net.layers[0], 1, 2, 1e-5
orig = layer.W[i, j]
layer.W[i, j] = orig + eps; loss_plus = 0.5 * np.mean((net.forward(x) - target) ** 2)
layer.W[i, j] = orig - eps; loss_minus = 0.5 * np.mean((net.forward(x) - target) ** 2)
layer.W[i, j] = orig
numerical_grad = (loss_plus - loss_minus) / (2 * eps)
print(f"analytic grad: {layer.dW[i, j]:.8f}   numerical grad: {numerical_grad:.8f}   "
      f"relative error: {abs(layer.dW[i, j] - numerical_grad) / abs(numerical_grad):.2e}")

## 6. Assembling the dataset

Each simulated particle carries an unknown conformational coordinate $t$ (the target we hope
the latent $\mathbf{z}$ recovers, used only for evaluation — never shown to the network), plus
a *known* in-plane pose and defocus (standing in for a prior consensus refinement's output).

In [ ]:
def make_dataset(n_particles, n=24, apix=3.0, noise_std=0.1, seed=0):
    """Simulate a particle stack with unknown conformational heterogeneity but
    *known* poses and CTF (mirroring real cryoDRGN's input: poses and CTF come
    from a prior consensus homogeneous refinement)."""
    rng = np.random.default_rng(seed)
    imgs = np.zeros((n_particles, n, n))
    hartleys = np.zeros((n_particles, n, n))
    Rs = np.zeros((n_particles, 3, 3))
    ctfs = np.zeros((n_particles, n, n))
    ts = np.zeros(n_particles)
    for i in range(n_particles):
        t = rng.uniform(0, 1)
        blobs = make_conformation(t)
        R = random_inplane_rotation(rng)
        k3 = slice_coords(n, apix, R)
        h_ideal = volume_hartley(k3, blobs)
        kmag = np.linalg.norm(k3[..., :2], axis=-1)
        df = rng.uniform(8000, 20000)
        c = ctf(kmag, defocus=df)
        h_obs = h_ideal * c
        img = hartley_to_real(h_obs)
        img = img / (np.abs(img).max() + 1e-8)
        noisy = img + rng.normal(scale=noise_std, size=img.shape)

        imgs[i] = noisy
        hartleys[i] = real_to_hartley(noisy)
        Rs[i] = R
        ctfs[i] = c
        ts[i] = t
    return {"imgs": imgs, "hartleys": hartleys, "Rs": Rs, "ctfs": ctfs, "ts": ts, "n": n, "apix": apix}


data = make_dataset(n_particles=1000, n=24, noise_std=0.1, seed=0)
print(f"dataset: {data['imgs'].shape[0]} particles, {data['n']}x{data['n']} px, apix={data['apix']}")

## 7. Training

The training step for one batch:

1. Encode each image's (whitened) Hartley coefficients to $(\mu, \log\sigma^2)$, sample
   $\mathbf{z}$ via reparameterization.
2. For every particle, take its *own* known central-slice coordinates, run them through the
   positional encoding, concatenate with (a copy of) that particle's $\mathbf{z}$, and decode
   — this gives one predicted Hartley coefficient per pixel of that particle's slice.
3. Multiply the predicted slice by that particle's known CTF and compare (MSE) to its observed
   Hartley-transformed image.
4. Backpropagate the reconstruction gradient through the decoder; the piece of that gradient
   landing on the (broadcast, repeated) $\mathbf{z}$ input is summed back across all pixels
   of that particle to get $\partial \mathcal{L}_{\text{recon}} / \partial \mathbf{z}$, which
   is then pushed through the reparameterization trick and the encoder — plus the closed-form
   KL gradient added directly at $(\mu, \log\sigma^2)$.

In [ ]:
def build_models(n_pix, zdim=1, n_freqs=6, hidden=96, seed=0):
    rng = np.random.default_rng(seed)
    encoder = MLP([
        Dense(n_pix * n_pix, hidden, rng), Tanh(),
        Dense(hidden, hidden, rng), Tanh(),
        Dense(hidden, 2 * zdim, rng),
    ])
    pe_dim = 3 + 3 * 2 * n_freqs
    decoder = MLP([
        Dense(pe_dim + zdim, hidden, rng), Tanh(),
        Dense(hidden, hidden, rng), Tanh(),
        Dense(hidden, 1, rng),
    ])
    return encoder, decoder


def train(data, zdim=1, n_freqs=6, hidden=96, epochs=100, batch_size=32, lr=2e-3, beta=0.02, seed=0):
    n = data["n"]
    n_particles = data["imgs"].shape[0]
    # Two different normalizations for two different purposes:
    #  - `enc_scale` (per-frequency std) whitens the encoder's *input* so no single
    #    low-frequency coefficient dominates its gradient signal.
    #  - `x_scale` (single global std) rescales the *reconstruction target* so the loss
    #    stays on a physically-meaningful, uniformly-weighted Hartley-coefficient scale
    #    (per-pixel target normalization would instead massively amplify noise at high
    #    frequency, where the true signal is near zero).
    enc_scale = data["hartleys"].std(axis=0, keepdims=True) + 1e-3
    x_scale = data["hartleys"].std()
    x_flat = (data["hartleys"] / enc_scale).reshape(n_particles, -1)

    encoder, decoder = build_models(n, zdim=zdim, n_freqs=n_freqs, hidden=hidden, seed=seed)
    rng = np.random.default_rng(seed + 1)

    coords = np.zeros((n_particles, n * n, 3))
    for i in range(n_particles):
        coords[i] = slice_coords(n, data["apix"], data["Rs"][i]).reshape(-1, 3)
    pos_enc_all = positional_encoding(coords, n_freqs=n_freqs)

    history = {"loss": [], "recon": [], "kl": []}
    n_batches = max(1, n_particles // batch_size)

    for epoch in range(epochs):
        order = rng.permutation(n_particles)
        ep_loss = ep_recon = ep_kl = 0.0
        for b in range(n_batches):
            idx = order[b * batch_size:(b + 1) * batch_size]
            B = len(idx)
            if B == 0:
                continue

            enc_out = encoder.forward(x_flat[idx])
            mu, logvar = enc_out[:, :zdim], enc_out[:, zdim:]
            std = np.exp(0.5 * logvar)
            eps = rng.normal(size=std.shape)
            z = mu + eps * std

            pe = pos_enc_all[idx].reshape(B * n * n, -1)
            z_rep = np.repeat(z, n * n, axis=0)
            pred = decoder.forward(np.concatenate([pe, z_rep], axis=1)).reshape(B, n, n)

            ctf_b = data["ctfs"][idx]
            pred_ctf = pred * ctf_b
            target = data["hartleys"][idx] / x_scale

            diff = pred_ctf - target
            recon_loss = np.mean(diff ** 2)
            kl_loss = np.mean(-0.5 * np.sum(1 + logvar - mu ** 2 - np.exp(logvar), axis=1))
            loss = recon_loss + beta * kl_loss

            d_pred = (diff * (2.0 / diff.size) * ctf_b).reshape(B * n * n, 1)
            d_dec_in = decoder.backward(d_pred)
            d_z = d_dec_in[:, -zdim:].reshape(B, n * n, zdim).sum(axis=1)

            d_mu = d_z + beta * mu / B
            d_logvar = d_z * (eps * 0.5 * std) + beta * (0.5 * (np.exp(logvar) - 1)) / B
            encoder.backward(np.concatenate([d_mu, d_logvar], axis=1))

            decoder.step(lr)
            encoder.step(lr)

            ep_loss += loss * B; ep_recon += recon_loss * B; ep_kl += kl_loss * B

        history["loss"].append(ep_loss / n_particles)
        history["recon"].append(ep_recon / n_particles)
        history["kl"].append(ep_kl / n_particles)

    return encoder, decoder, history, x_scale, enc_scale


encoder, decoder, history, x_scale, enc_scale = train(
    data, zdim=1, n_freqs=6, hidden=96, epochs=100, batch_size=32, lr=2e-3, beta=0.02, seed=0
)
print(f"final loss={history['loss'][-1]:.4f}  recon={history['recon'][-1]:.4f}  kl={history['kl'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, key, title in zip(axes, ["loss", "recon", "kl"], ["total ELBO loss", "reconstruction term", "KL term"]):
    ax.plot(history[key], color="#4fc3f7")
    ax.set_xlabel("epoch"); ax.set_title(title)
plt.tight_layout(); plt.show()

## 8. Did it work? Checking the latent against the (held-out) ground truth

We never showed the network $t$ during training — it only ever saw noisy, CTF-corrupted
images. We now encode every particle and check whether the learned 1D latent $\mathbf{z}$
correlates with the true conformational coordinate $t$. (The *sign* of the correlation is
arbitrary — nothing in the VAE objective prefers $\mathbf{z}$ increasing with $t$ over
decreasing with it — so we report the unsigned strength.)

In [ ]:
n_particles = data["imgs"].shape[0]
x_flat_eval = (data["hartleys"] / enc_scale).reshape(n_particles, -1)
mu_all = encoder.forward(x_flat_eval)[:, 0]

pear = pearsonr(mu_all, data["ts"])[0]
spear = spearmanr(mu_all, data["ts"])[0]
print(f"Pearson correlation(z, true t):  {pear:+.4f}")
print(f"Spearman correlation(z, true t): {spear:+.4f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
sc = ax.scatter(data["ts"], mu_all, c=data["ts"], cmap="cool", s=10, alpha=0.6)
ax.set_xlabel("true conformational coordinate t"); ax.set_ylabel("learned latent z")
ax.set_title(f"latent vs. ground truth  (Pearson r = {pear:+.3f})")
plt.tight_layout(); plt.show()

## 9. Decoding volumes across the recovered latent space

The payoff of a coordinate-based decoder: once trained, we can query it at **any** latent
value and **any** 3D coordinate grid we like — including a canonical, un-rotated view — to
generate an explicit output image/volume at that point in conformation space. Sweeping across
the recovered latent range should retrace the blob's sliding motion, entirely from the noisy
input images.

In [ ]:
n, apix = data["n"], data["apix"]
coords_canon = slice_coords(n, apix, np.eye(3)).reshape(-1, 3)
pe_canon = positional_encoding(coords_canon, n_freqs=6)

zmin, zmax = np.percentile(mu_all, [2, 98])
z_values = np.linspace(zmin, zmax, 6)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, zval in zip(axes, z_values):
    z_rep = np.full((pe_canon.shape[0], 1), zval)
    h_pred = decoder.forward(np.concatenate([pe_canon, z_rep], axis=1)).reshape(n, n)
    ax.imshow(hartley_to_real(h_pred), cmap="gray")
    ax.set_title(f"z = {zval:.2f}")
    ax.axis("off")
plt.suptitle("Decoded structure across the recovered latent range", y=1.05)
plt.tight_layout(); plt.show()

## 10. What would we have seen without cryoDRGN?

A classical, homogeneous reconstruction pipeline assumes a single rigid structure and
averages all particles together (after aligning them to a common orientation). If the
particles are secretly a mixture of conformations, this average **blurs across every state
that was averaged in** — exactly the information loss cryoDRGN is designed to recover from.

In [ ]:
rng2 = np.random.default_rng(2)
n_states = 400
ts_mix = rng2.uniform(0, 1, n_states)
clean_stack = np.zeros((n_states, n, n))
coords_id = slice_coords(n, apix, np.eye(3))
for i, t in enumerate(ts_mix):
    h = volume_hartley(coords_id, make_conformation(t))
    clean_stack[i] = hartley_to_real(h)
naive_avg = clean_stack.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
axes[0].imshow(naive_avg, cmap="gray")
axes[0].set_title("naive average over\nall conformations (blurred)")
for ax, t in zip(axes[1:], [0.0, 1.0]):
    h = volume_hartley(coords_id, make_conformation(t))
    ax.imshow(hartley_to_real(h), cmap="gray")
    ax.set_title(f"true single state, t={t}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

The naive average smears blob B's contribution across its entire range of motion into a
faint vertical streak — a classic sign of unresolved heterogeneity in a real cryo-EM
reconstruction. CryoDRGN's decoder, conditioned on the recovered latent code, instead
retraces the two distinct end states and everything in between.

## 11. Running real cryoDRGN on real data

This notebook trains everything from scratch on a fully synthetic dataset for the sake of
CPU-only reproducibility. On a real EMPIAR dataset, the workflow (with the official
[`cryodrgn`](https://github.com/ml-struct-bio/cryodrgn) package, GPU required) looks like:

```bash
# 1. Start from a consensus (homogeneous) refinement's outputs: particle stack,
#    poses, and CTF parameters (e.g. from RELION or cryoSPARC).
cryodrgn parse_pose_star particles.star -o poses.pkl
cryodrgn parse_ctf_star particles.star -o ctf.pkl

# 2. Train the heterogeneous reconstruction VAE
cryodrgn train_vae particles.mrcs --ctf ctf.pkl --poses poses.pkl \
    --zdim 8 -n 50 -o outputs/

# 3. Explore the learned latent space and generate volumes
cryodrgn analyze outputs/ 49          # PCA/UMAP of the latent space, clustering
cryodrgn eval_vol outputs/weights.49.pkl --config outputs/config.yaml \
    -z <latent-coordinate> -o volume.mrc
```

The encoder/decoder architectures, positional encoding, and Hartley-domain, CTF-weighted ELBO
in the real package are the same ideas implemented above, just with `torch.nn` + autograd, a
CNN-friendlier input representation, full random-$SO(3)$ pose support, and much larger
default network capacity.

## 12. Discussion questions

1. We saw that switching from in-plane-only poses to full random $SO(3)$ poses broke latent
   recovery at this dataset size. What does that imply about the *minimum* information content
   per particle needed to jointly disentangle pose and conformation — and why might increasing
   `n_freqs` (positional encoding bands) or `hidden` (decoder capacity) *not* fix it on its own?
2. Our toy molecule has exactly one continuous degree of freedom, so `zdim=1` was the right
   choice. Real biomolecules can have many independent modes of motion. What would you expect
   to go wrong if you trained with `zdim=1` on a molecule with two *independent* conformational
   coordinates? What diagnostic (only using the trained model, not ground truth) would reveal it?
3. CryoDRGN uses a coordinate-based decoder instead of a fixed voxel grid. What does this buy
   you at *training* time (think about what the loss can be computed on) versus at *inference/
   analysis* time (think about what resolution or region you can query)?
4. The naive homogeneous average in Section 10 used *clean* projections (no noise, no CTF) so
   the blurring we see is purely due to heterogeneity. If you instead averaged the noisy,
   CTF-corrupted images directly (no CTF correction), what *additional* artifact would you
   expect on top of the heterogeneity blur, and why?
5. Our reconstruction loss weights every Hartley coefficient equally after a single global
   rescaling. Real cryo-EM signal-to-noise ratio falls off sharply at high spatial frequency.
   How might a frequency-dependent loss weighting change what the model learns to prioritize,
   and would you expect it to help or hurt the latent's identifiability of $t$?
6. Compositional heterogeneity (a subunit present/absent) versus conformational heterogeneity
   (continuous motion, as in this notebook) both fall out of the same VAE framework — the
   difference is entirely in what the *training data's* underlying distribution of true states
   looks like. Sketch how you'd modify `make_conformation` to generate a compositional-only
   heterogeneity dataset, and what you'd expect the latent space to look like after training
   (continuum vs. discrete clusters).

## References

- Zhong, E.D., Bepler, T., Berger, B. & Davis, J.H. CryoDRGN: reconstruction of heterogeneous
  cryo-EM structures using neural networks. *Nat. Methods* **18**, 176–185 (2021).
- Zhong, E.D., Bepler, T., Davis, J.H. & Berger, B. Reconstructing continuous distributions of
  3D protein structure from cryo-EM images. *ICLR* (2020).
- Mildenhall, B. et al. NeRF: Representing scenes as neural radiance fields for view synthesis.
  *ECCV* (2020). (Source of the positional-encoding trick used in the decoder.)
- `cryodrgn` codebase and documentation: https://github.com/ml-struct-bio/cryodrgn